## EDA: Justificación

Se justifica la viabilidad de las técnicas y lo qué se espera hallar con cada una

---
### **1. Factibilidad**

#### a. Clustering y anomalías

- **86,017 proveedores y 2,996 entidades** con features agregables (17 y 14 features)

- Montos con cola pesada (mediana S/192k vs. p99 S/23.1M)

- **Mediana de 1 adjudicación por proveedor:** la mayoría tiene un vector casi vacío

- KMeans++ escala sobre todos los proveedores (Spark MLlib); DBSCAN/HDBSCAN y jerárquico son O(n²)

- Sobre **todas las entidades** es viable y usar una **muestra estratificada** de proveedores

- **Señales:** 36.8% de entidades con HHI > 0.25; métodos con 82-98% de postor único

#### b. Sistemas de recomendación

- Matriz **entidad × proveedor: 2,992 × 86,017, densidad 0.053%**

- Matriz **entidad × familia: 2,992 × 386, densidad 3.4%**

- Contenido: `descripcion` / `items_desc` por proveedor → TF-IDF

- 36 meses continuos, con `anio` en cada celda para evitar fuga

- **Salvedad:** el 84.9% de proveedores tiene una sola entidad

#### c. Análisis de enlaces: PageRank y HITS

- **Bipartito dirigido entidad → proveedor: 89,009 nodos y 136,099 aristas**

- Grado con cola pesada (CCDF casi recta en log-log): la centralidad discrimina entre actores

- HITS encaja con la estructura: hubs = entidades, authorities = proveedores

#### d. Grafos y detección de comunidades

- **Co-participación: 60,213 proveedores y 1.64M aristas** (tope 30 participantes, ≥ 2 procesos juntos)

- Pares que compiten juntos más de 2,500 veces: estructura de mercado clara

- **Hubs extremos** (grado p99 = 12,059 vs. mediana 16): hay que filtrarlos o normalizarlos

- 10.9% de proveedores en varias regiones y 9.9% en varios segmentos

#### e. Aprendizaje de representación en grafos

- **SVD truncada** de la matriz entidad × proveedor (136k celdas): embedding directo de ambos tipos de nodo

- **Node2Vec** sobre co-participación (1.6M aristas), sin hubs o en un subgrafo

- Visualización t-SNE/UMAP contra clústeres y comunidades, y **predicción de enlaces**

---

### **2. Objetivos**

#### a. Clustering y detección de anomalías

- **Qué se busca:** tipologías de entidades y de proveedores 

- Grandes generalistas, locales de un solo cliente y obras con postor único, y los actores que no encajan en ninguna

- **Anomalías:** actores lejos de su centroide o marcados como ruido por DBSCAN/HDBSCAN p

- Por ejemplo mucho monto con poca competencia y un solo cliente

- **Preguntas:** PQ1 (segmento cautivo) y PQ2 (perfiles de mercado)

- **Artefacto:** `features_proveedor`, `features_entidad`

#### b. Sistemas de recomendación

- **Qué se busca:** para cada entidad, proveedores **candidatos** que entidades parecidas contratan y ella no. 

- Entre ellos, **sustitutos** de sus proveedores actuales (mismo rubro y vector parecido)

- **Relaciones atípicas:** relaciones existentes con puntaje muy bajo, que el patrón general no explica

- **Uso descriptivo:** ¿la dependencia de una entidad cautiva se explica por falta de alternativas?

- **Preguntas:** PQ3, y PQ1 como complemento

- **Artefacto:** `matriz_entidad_proveedor`, `matriz_entidad_familia`, texto de `procesos`

#### c. Análisis de enlaces: PageRank y HITS

- **Qué se busca:** proveedores **centrales** (reciben de muchas entidades) y entidades **hub** (compran a proveedores centrales)

- **Contraste:** centralidad vs. monto; un proveedor central con pocas adjudicaciones, o al revés, es llamativo

- **Pregunta:** PQ1

- **Artefacto:** `grafo_bipartito`

#### d. Grafos y detección de comunidades

- **Qué se busca:** **mercados** de proveedores que compiten entre sí, y si se organizan por región o por rubro

- **Solapamiento:** proveedores que viven en varios mercados (BigCLAM)

- **Grupos cerrados:** comunidades pequeñas y densas que compiten siempre juntas

- **Pregunta:** PQ2

- **Artefacto:** `grafo_coparticipacion`

#### e. Aprendizaje de representación en grafos

- **Qué se busca:** embeddings de entidades y proveedores (SVD de la matriz, Node2Vec del grafo)

- **Uso:** visualizarlos (t-SNE/UMAP) contra los clústeres y las comunidades

- Predecir enlaces nuevos de 2025 entrenando con 2023-2024

- **Preguntas:** PQ2, PQ3

- **Artefacto:** `matriz_entidad_proveedor`, `grafo_coparticipacion`

---
### **3. Resumen**

| Técnica | Factible | Pregunta | Artefacto | Salvedad |
|---|---|---|---|---|
| Clustering y anomalías | Sí | PQ1, PQ2 | `features_proveedor`, `features_entidad` | O(n²) en muestra |
| Recomendación | Sí | PQ3 | `matriz_entidad_proveedor`, `matriz_entidad_familia` | 84.9% prov con 1 entidad |
| PageRank / HITS | Sí | PQ1 | `grafo_bipartito` | — |
| Comunidades | Submuestra | PQ2 | `grafo_coparticipacion` | Hubs extremos y subgrafo |
| Representación | Sí | PQ2, PQ3 | matriz + `grafo_coparticipacion` | Node2Vec sin hubs |